In [1]:
from langchain_core.documents import Document

In [2]:
#!pip install -U langchain langchain-openai langchain-gemini langchain-groq chromadb tiktoken transformers accelerate bitsandbytes sentence-transformers

In [3]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from dotenv import dotenv_values

env_vars = dotenv_values("../.env")
os.environ.update(env_vars)

d:\langchain\rag_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
#!pip install langchain langchain-community pypdf


## Load the Document

In [5]:
# from langchain_community.document_loaders import PyPDFLoader

# loader = PyPDFLoader("../data/sunaina_AIML_resume.pdf")
# documents = loader.load()


In [6]:
# print(f"Loaded {len(documents)} documents")

In [24]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/RAG_Notes.txt")
documents = loader.load()
print(f"Loaded {len(documents)} documents")


Loaded 1 documents


In [25]:
print(type(documents))
print(len(documents))
print(documents[0])
print(documents[0].page_content)

<class 'list'>
1
page_content='RAG (Retrieval-Augmented Generation) â€“ Explained Simply

1. What is RAG?
RAG stands for Retrieval-Augmented Generation.
It is a technique that combines:
- Information Retrieval (searching relevant data)
- Text Generation (using Large Language Models like GPT)

Instead of relying only on the modelâ€™s training data, RAG fetches relevant external information at query time and then generates an answer using that information.

--------------------------------------------------

2. Why RAG is Needed
Normal LLMs:
- Have fixed knowledge (limited to training data)
- Can hallucinate (generate incorrect information)
- Cannot access private or updated data

RAG solves this by:
- Using external documents
- Providing fresh, accurate, and domain-specific answers

--------------------------------------------------

3. How RAG Works (Step-by-Step)

Step 1: Document Loading
- PDFs, text files, web pages, or databases are loaded.

Step 2: Chunking
- Large documents are s

## Chunking

Use RecursiveCharacterTextSplitter

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

chunk_size=500 → approx words/characters

chunk_overlap=100 → helps context continuity

In [27]:
print(len(chunks))
print(chunks[0].page_content)


5
RAG (Retrieval-Augmented Generation) â€“ Explained Simply

1. What is RAG?
RAG stands for Retrieval-Augmented Generation.
It is a technique that combines:
- Information Retrieval (searching relevant data)
- Text Generation (using Large Language Models like GPT)

Instead of relying only on the modelâ€™s training data, RAG fetches relevant external information at query time and then generates an answer using that information.

--------------------------------------------------

2. Why RAG is Needed
Normal LLMs:
- Have fixed knowledge (limited to training data)
- Can hallucinate (generate incorrect information)
- Cannot access private or updated data

RAG solves this by:
- Using external documents
- Providing fresh, accurate, and domain-specific answers


🧠 Visual Flow 
```
PDF
 ↓
Pages
 ↓
Chunks
```

In [11]:
!pip install faiss-cpu

## Import Gemini Embeddings

In [12]:
from langchain_community.embeddings import SentenceTransformerEmbeddings

In [28]:
#Create embedding object:

embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

## Create Vector Store (FAISS)

In [29]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)


Similarity Search (THE MAGIC ✨)

In [30]:
query = "Explain RAG in simple terms"
results = vectorstore.similarity_search(query, k=2)



In [31]:
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(doc.page_content)
    print("-" * 40)


Result 1:
RAG (Retrieval-Augmented Generation) â€“ Explained Simply

1. What is RAG?
RAG stands for Retrieval-Augmented Generation.
It is a technique that combines:
- Information Retrieval (searching relevant data)
- Text Generation (using Large Language Models like GPT)

Instead of relying only on the modelâ€™s training data, RAG fetches relevant external information at query time and then generates an answer using that information.

--------------------------------------------------

2. Why RAG is Needed
Normal LLMs:
- Have fixed knowledge (limited to training data)
- Can hallucinate (generate incorrect information)
- Cannot access private or updated data

RAG solves this by:
- Using external documents
- Providing fresh, accurate, and domain-specific answers
----------------------------------------
Result 2:
RAG solves this by:
- Using external documents
- Providing fresh, accurate, and domain-specific answers

--------------------------------------------------

3. How RAG Works (Ste

In [32]:
query1 = "What are the benefits of RAG?"
results1 = vectorstore.similarity_search(query1, k=2)

for i, doc in enumerate(results1):
    print(f"Result1 {i+1}:")
    print(doc.page_content)
    print("-" * 40) 


Result1 1:
RAG (Retrieval-Augmented Generation) â€“ Explained Simply

1. What is RAG?
RAG stands for Retrieval-Augmented Generation.
It is a technique that combines:
- Information Retrieval (searching relevant data)
- Text Generation (using Large Language Models like GPT)

Instead of relying only on the modelâ€™s training data, RAG fetches relevant external information at query time and then generates an answer using that information.

--------------------------------------------------

2. Why RAG is Needed
Normal LLMs:
- Have fixed knowledge (limited to training data)
- Can hallucinate (generate incorrect information)
- Cannot access private or updated data

RAG solves this by:
- Using external documents
- Providing fresh, accurate, and domain-specific answers
----------------------------------------
Result1 2:
RAG solves this by:
- Using external documents
- Providing fresh, accurate, and domain-specific answers

--------------------------------------------------

3. How RAG Works (S

## Create a Retriever

In [33]:
#From your FAISS vectorstore:

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

k=3 → fetch top 3 relevant chunks

Retriever = smart search layer

## Create RAG Prompt Template

In [34]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
You are a helpful assistant.
Answer the question using ONLY the context below.
If the answer is not present, say "I don't know".

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


Load Gemini LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

temperature=0 → factual, not creative

Manual RAG Flow

1️⃣ Retrieve relevant chunks

In [36]:
question = "What are the benefits of RAG?"

docs = retriever.invoke(question)

2️⃣ Combine context

In [37]:
context = "\n\n".join([doc.page_content for doc in docs])


3️⃣ Generate answer using Gemini

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

response = llm.invoke(
    prompt.format(context=context, question=question)
)

print(response.content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-pro' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-pro is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}